<a href="https://colab.research.google.com/github/dimitrod/ehu_nlp_dimathina/blob/Huggingface_tutorials/Huggingface_SimpleRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
!pip install langchain langchain-openai langchain-community faiss-cpu tiktoken datasets bitsandbytes transformers sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 18.4 MB/s eta 0:00:00


In [30]:
from langchain_openai import OpenAI
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
#from langchain_community.vectorstores import FAISS
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_core.vectorstores import VectorStoreRetriever
from langchain.chains import RetrievalQA
from datasets import load_dataset
import os

In [3]:
%time trivia_qa_wikipedia = load_dataset('trivia_qa', name="rc.wikipedia")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/26.7k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

train-00000-of-00007.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

train-00001-of-00007.parquet:   0%|          | 0.00/261M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/221M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61888 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7993 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7701 [00:00<?, ? examples/s]

CPU times: user 17.1 s, sys: 14 s, total: 31.1 s
Wall time: 58.3 s


In [4]:
print(trivia_qa_wikipedia)

DatasetDict({
    train: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 61888
    })
    validation: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 7993
    })
    test: Dataset({
        features: ['question', 'question_id', 'question_source', 'entity_pages', 'search_results', 'answer'],
        num_rows: 7701
    })
})


In [65]:
training_set = trivia_qa_wikipedia["train"]["entity_pages"]

In [90]:
len(training_set[7090]["wiki_context"])

5

In [74]:
len(training_set)

61888

In [78]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    length_function=len,
)

In [99]:
train_documents = []
i = 0

for docs in training_set :
  if i == 100 :
    break
  i += 1
  print(f"loading document {i} of {len(training_set)}: ", docs["title"])

  temp = ""
  for doc in docs["wiki_context"] :
    temp += doc

  texts = text_splitter.create_documents([temp])
  print(f"number of chunks: {len(texts)}")
  for text in texts :
    text.metadata["source"] = docs["title"]
    train_documents.append(text)

loading document 1 of 61888:  ['England', 'Judi Dench']
number of chunks: 360
loading document 2 of 61888:  ['Nation state', 'Angola', 'Angolan Civil War']
number of chunks: 476
loading document 3 of 61888:  ['David Soul']
number of chunks: 27
loading document 4 of 61888:  ['Super Bowl XX']
number of chunks: 70
loading document 5 of 61888:  ['Ethnic groups in Europe', 'Capital punishment']
number of chunks: 285
loading document 6 of 61888:  ['Integrated Services Digital Network']
number of chunks: 79
loading document 7 of 61888:  ['Bruce Willis']
number of chunks: 70
loading document 8 of 61888:  ['Lord of the Flies']
number of chunks: 28
loading document 9 of 61888:  ['Joan Rivers']
number of chunks: 89
loading document 10 of 61888:  ['Patricia Neary']
number of chunks: 5
loading document 11 of 61888:  ['Europe']
number of chunks: 186
loading document 12 of 61888:  ['Vought-Sikorsky VS-300']
number of chunks: 10
loading document 13 of 61888:  ['Joseph Goebbels']
number of chunks: 151


In [100]:
len(train_documents)

14406

In [101]:
train_documents[14300]

Document(metadata={'source': ['Cry, the Beloved Country', 'South Africa']}, page_content='Climate change is expected to bring considerable warming and drying to much of this already semi-arid region, with greater frequency and intensity of extreme weather events such as heatwaves, flooding and drought. According to computer generated climate modelling produced by the South African National Biodiversity Institute  parts of southern Africa will see an increase in temperature by about one degree Celsius along the coast to more than four degrees Celsius in the already hot hinterland')

In [46]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    length_function=len,
)

In [77]:
docs = []

for doc in train_documents :
  temp = []
  for d in doc :
    temp = text_splitter.create_documents([d])

  for t in temp :
    docs.append(t)

KeyboardInterrupt: 

In [51]:
print(len(docs))

for d in docs[7801]:
  print(d)

9501
('id', None)
('metadata', {})
('page_content', '"The Junior League Baseball Division is a program for boys and girls ages 13–14, using a conventional 90 ft diamond with a pitching distance of 60 ft. (A modified diamond is available during the regular season.) The local league has an option to choose a Tournament Team (or "All Stars") of 13-14-year-olds from within this division (and/or from within the Senior League Division), and the team may enter the International Tournament. The culmination of the International Tournament is the Junior')
('type', 'Document')


In [50]:
db = FAISS.from_documents(docs, HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5"))

In [43]:
query = "Who won the Super Bowl XX?"
answer = db.similarity_search(query)
print(answer[1])

page_content='As of 2016, Super Bowl XX remains the last Super Bowl to feature two teams both making their first appearance in the game. It was the fourth overall following Super Bowl I, Super Bowl III, and Super Bowl XVI. Any future Super Bowl that would have such a combination would have to have the Detroit Lions playing either the Cleveland Browns, Houston Texans, or Jacksonville Jaguars in the game.'


In [35]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# **LLM Chain starts here**

In [55]:
pip install -U bitsandbytes

In [57]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "HuggingFaceH4/zephyr-7b-beta"

#bnb_config = BitsAndBytesConfig(
#    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
#)

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00008.safetensors:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

model-00002-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

model-00003-of-00008.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

model-00004-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [58]:
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from transformers import pipeline
from langchain_core.output_parsers import StrOutputParser

text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=True,
    max_new_tokens=400,
)

llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

prompt_template = """
<|system|>
Answer the question based on your knowledge. Use the following context to help:

{context}

</s>
<|user|>
{question}
</s>
<|assistant|>

 """

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template,
)

llm_chain = prompt | llm | StrOutputParser()

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
<ipython-input-58-3303a33a3a94>:17: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_generation_pipeline)


In [59]:
from langchain_core.runnables import RunnablePassthrough

retriever = db.as_retriever()

rag_chain = {"context": retriever, "question": RunnablePassthrough()} | llm_chain

In [60]:
question = "What age do players in the Junior League Baseball Division usually have?"

In [61]:
llm_chain.invoke({"context": "", "question": question})

"\n<|system|>\nAnswer the question based on your knowledge. Use the following context to help:\n\n\n\n</s>\n<|user|>\nWhat age do players in the Junior League Baseball Division usually have?\n</s>\n<|assistant|>\n\n  Players in the Junior League Baseball division typically range in age from 13 to 16 years old, depending on the specific league's age cutoff. This division is designed for older youth baseball players who have outgrown the Little League Baseball division but are not yet ready for the more advanced play of high school or travel ball leagues. The focus in this division is on developing fundamental skills, sportsmanship, and teamwork, as well as preparing players for higher levels of competition."

In [63]:
rag_chain.invoke(question)

'\n<|system|>\nAnswer the question based on your knowledge. Use the following context to help:\n\n[Document(metadata={}, page_content="The Minor League Baseball division is generally for children ages 7–11, with local leagues given the option to allow 6-year-old children to try out. Local leagues are permitted to further divide the Minor League division based on player age and/or experience, and often consist of coach-pitch (i.e., the batter\'s coach lightly pitching the ball) or machine-pitch at lower levels, with defensive players pitching at higher levels.\\n\\n9–10 Year Olds"), Document(metadata={}, page_content=\'"The Junior League Baseball Division is a program for boys and girls ages 13–14, using a conventional 90 ft diamond with a pitching distance of 60 ft. (A modified diamond is available during the regular season.) The local league has an option to choose a Tournament Team (or "All Stars") of 13-14-year-olds from within this division (and/or from within the Senior League Div